# 🌐 100M Parameter Multilingual GPT (Bangla + English + Math)
## 4096 Context Length (4k Context) | 10,000-Vocab BPE | GQA 3:1 | RoPE

এই নোটবুকটি দিয়ে Google Colab Free T4 GPU-তে সম্পূর্ণ স্ক্র্যাচ থেকে অথবা কন্টিন্যুয়াল পদ্ধতিতে **১০ কোটি (100M) প্যারামিটারের বহুভাষিক বেস মডেল** তৈরি ও ট্রেনিং করা যাবে।

### 🌟 মডেল আর্কিটেকচার স্পেসিফিকেশন:
- **প্যারামিটার সংখ্যা:** ~১০০ Million (95.78M Tied / 103.46M Untied)
- **কনটেক্সট লেন্থ:** **৪০৯৬ টোকেন (4k Context)** — একসাথে প্রায় ৩,২০০ পূর্ণাঙ্গ শব্দ ধারণক্ষমতা!
- **ভোকাবুলারি:** **১০,০০০ (10k)** BPE সাবওয়ার্ড (বাংলা + ইংরেজি + গণিত ও যুক্তি)
- **হিডেন ডাইমেনশন:** ৭৬৮ | **লেয়ার সংখ্যা:** ১৪ ট্রান্সফরমার ব্লক
- **অ্যাটেনশন:** GQA (১২ কুয়েরি হেড, ৪ কেভি হেড, ৩:১ অনুপাত — মোবাইল মেমোরি ৬৬% সাশ্রয়ী!)
- **পজিশনাল এমবেডিং:** RoPE (Rotary Position Embeddings, Base: 10000.0)
- **মেমরি সুরক্ষা:** NanoGPT-স্টাইল `uint16 memmap` চাঙ্কড স্ট্রিমিং — মাত্র ৫০ MB RAM খরচ!


## ✅ ধাপ ১: GPU ও হার্ডওয়্যার যাচাই


In [ ]:
import torch, sys
print('=' * 60)
print('🖥️  হার্ডওয়্যার ও জিপিইউ তথ্য')
print('=' * 60)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram = gpu.total_memory / 1024**3
    print(f'✓ GPU: {gpu.name}')
    print(f'✓ VRAM: {vram:.1f} GB')
    print('✓ ১০০M মডেল (৪০৯৬ কনটেক্সট) রান করার জন্য হার্ডওয়্যার সম্পূর্ণ উপযুক্ত!')
else:
    print('⚠️  GPU নেই! Runtime > Change runtime type > T4 GPU সিলেক্ট করুন।')
print(f'Python: {sys.version[:6]} | PyTorch: {torch.__version__}')
print('=' * 60)


## 📦 ধাপ ২: প্রয়োজনীয় প্যাকেজ ইনস্টল


In [ ]:
!pip install -q tokenizers>=0.14.0 sentencepiece datasets
print("✓ সকল প্রয়োজনীয় লাইব্রেরি ইনস্টল সম্পন্ন!")


## 💾 ধাপ ৩: Google Drive মাউন্ট ও ডিরেক্টরি সেটআপ


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/ss_100million/src', exist_ok=True)
os.makedirs('/content/ss_100million/data', exist_ok=True)
os.makedirs('/content/ss_100million/checkpoints', exist_ok=True)

DRIVE_CKPT_100M = '/content/drive/MyDrive/bengali_gpt_100m_checkpoints'
os.makedirs(DRIVE_CKPT_100M, exist_ok=True)
open('/content/ss_100million/src/__init__.py', 'a').close()
print("✓ ১০০M ডিরেক্টরি সফলভাবে সেটআপ হয়েছে!")


## 📚 ধাপ ৪: ট্রেনিং কর্পাস প্রস্তুতি (মূল ১০k বেস কর্পাস + ইন্টারনেট থেকে নতুন ডেটা)

### 💡 এই ধাপে কী ঘটবে?
1. আপনি আপনার পিসির `ss_100million/data/corpus.txt` ফাইলটি আপনার গুগল ড্রাইভের মূল ডিরেক্টরিতে আপলোড করবেন: `/content/drive/MyDrive/corpus.txt`
   - এটিই আমাদের **মূল ১০,০০০ ভোকাব সমন্বিত বেস কর্পাস (বাংলা + English + Math)**।
2. নিচের সেলটি রান করলে ড্রাইভের সেই মূল ফাইলের সাথে ইন্টারনেট (Hugging Face) থেকে সরাসরি **আরও ২০,০০০ বাংলা উইকিপিডিয়া নিবন্ধ এবং ১০,০০০ ইংরেজি নিবন্ধ** ডাউনলোড করে শেষে যুক্ত (Append) করে দেবে!
3. ফলাফল: **কর্পাস = মূল 10k বেস ডেটা + নতুন ডাউনলোড করা বিশাল ডেটা** — যা দিয়ে প্রথম ১০০M বেস মডেল ট্রেন হবে!


In [ ]:
from datasets import load_dataset
import os

CORPUS_PATH = '/content/drive/MyDrive/corpus.txt'
print('=' * 65)
print('📖 ট্রেনিং কর্পাস প্রস্তুতি (মূল 10k বেস + নতুন ডাউনলোডকৃত ডেটা)')
print('=' * 65)

# ১. মূল আপলোড করা corpus.txt যাচাই
if not os.path.exists(CORPUS_PATH):
    print(f'⚠️ সতর্কবার্তা: {CORPUS_PATH} পাওয়া যায়নি!')
    print('👉 অনুগ্রহ করে পিসি থেকে ss_100million/data/corpus.txt ফাইলটি গুগল ড্রাইভের MyDrive-এ আপলোড করুন।')
    print('👉 অথবা নিচের কোড সরাসরি নতুন করে ডেটা ডাউনলোড করে ড্রাইভের corpus.txt তৈরি করবে।')
    with open(CORPUS_PATH, 'w', encoding='utf-8') as f:
        f.write('[100M MULTILINGUAL & MATH BASE CORPUS]\n\n')

base_size_mb = os.path.getsize(CORPUS_PATH) / (1024 * 1024)
print(f'✓ মূল বেস কর্পাস ফাইলের আকার: {base_size_mb:.2f} MB')

# ২. ইন্টারনেট থেকে আরও বাংলা ও ইংরেজি ডেটা ডাউনলোড করে শেষে যুক্ত করা (Append)
BN_ARTICLES = 15000
EN_ARTICLES = 8000

print('\n📡 ইন্টারনেট থেকে নতুন ডেটা ডাউনলোড করে মূল ফাইলে যুক্ত হচ্ছে...')
try:
    bn_ds = load_dataset('wikimedia/wikipedia', '20231101.bn', split='train')
except Exception:
    bn_ds = load_dataset('sagorsarker/bangla-wikipedia', split='train')

with open(CORPUS_PATH, 'a', encoding='utf-8') as f:
    f.write('\n\n[ADDITIONAL EXPANSION: BANGLA WIKIPEDIA]\n\n')
    for i, row in enumerate(bn_ds):
        if i >= BN_ARTICLES: break
        t = row['text'].strip()
        if len(t) > 120:
            f.write(t + '\n\n')
        if (i+1) % 5000 == 0:
            print(f'  ✍️  {i+1:,}/{BN_ARTICLES:,} বাংলা নিবন্ধ যুক্ত হয়েছে...')

try:
    en_ds = load_dataset('wikimedia/wikipedia', '20231101.simple', split='train')
    with open(CORPUS_PATH, 'a', encoding='utf-8') as f:
        f.write('\n\n[ADDITIONAL EXPANSION: ENGLISH KNOWLEDGE]\n\n')
        for i, row in enumerate(en_ds):
            if i >= EN_ARTICLES: break
            t = row['text'].strip()
            if len(t) > 120:
                f.write(t + '\n\n')
            if (i+1) % 4000 == 0:
                print(f'  ✍️  {i+1:,}/{EN_ARTICLES:,} ইংরেজি নিবন্ধ যুক্ত হয়েছে...')
except Exception as e:
    print('  ⚠️ English ডাউনলোড স্কিপ হয়েছে:', e)

total_mb = os.path.getsize(CORPUS_PATH) / (1024 * 1024)
print('\n' + '=' * 65)
print('✅ চূড়ান্ত কম্বাইন্ড কর্পাস প্রস্তুত (10k Base + New Downloaded Data)!')
print(f'• ফাইল পাথ: {CORPUS_PATH}')
print(f'• মোট আকার: {total_mb:.1f} MB')
print(f'• আনুমানিক টোকেন: ~{int(total_mb * 600_000):,}')
print('=' * 65)


## 🔧 ধাপ ৫: ১০০M সোর্স কোড ও মডিউল তৈরি (4k Context + 10k Vocab + Memmap)


In [ ]:
%%writefile /content/ss_100million/src/__init__.py


In [ ]:
%%writefile /content/ss_100million/src/config.py
"""
Configuration for 100M Parameter Multilingual (Bangla + English + Math) GPT Model.
Features: 4096 Context Length (4k Context), GQA (Grouped-Query Attention 3:1), RoPE, and 10k Vocab.
"""

import torch


class GPTConfig:
    """Model & Training hyperparameters for 100M Parameter Model with 4096 context length."""

    # Architecture (~96M to 103.5M parameters)
    vocab_size    = 10000      # 10,000-Vocab BPE Subword Tokenizer (Bangla + English + Math)
    embed_dim     = 768        # Hidden dimension
    num_heads     = 12         # Query attention heads (768 / 12 = 64 head_dim)
    num_kv_heads  = 4          # Key-Value heads (GQA 3:1 ratio -> cuts mobile KV-Cache by 66.7%!)
    num_layers    = 14         # 14 Transformer layers
    intermediate_dim = 3072    # 4x MLP expansion (768 * 4)
    block_size    = 4096       # 4096 Context Length (4k Context! holds ~3,200 words!)
    dropout       = 0.1
    bias          = False

    # RoPE (Rotary Position Embeddings)
    rope_theta    = 10000.0    # Base frequency for RoPE

    # Training Hyperparameters (Tuned for Colab Free T4 GPU)
    learning_rate = 3e-4
    min_lr        = 3e-5
    batch_size    = 4          # Micro-batch size (4 sequences of 4096 tokens)
    gradient_accumulation_steps = 4  # Effective batch size = 16 (4 * 4)
    max_iters     = 5000       # Total training iterations
    warmup_iters  = 250        # Warmup steps
    eval_interval = 500        # Evaluate every 500 steps
    eval_iters    = 20
    weight_decay  = 0.1
    beta1, beta2  = 0.9, 0.95
    grad_clip     = 1.0

    # LoRA fine-tuning parameters (Scaled for 100M)
    lora_rank           = 16
    lora_alpha          = 32
    lora_dropout        = 0.05
    lora_target_modules = ['q_proj', 'v_proj']

    # Checkpoint settings (Drive synced)
    checkpoint_dir  = 'checkpoints'
    save_interval   = 500

    # System & Hardware
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed   = 42

    # Generation defaults
    temperature    = 0.75
    top_k          = 40
    max_new_tokens = 250

    @classmethod
    def estimate_parameters(cls):
        tok_emb = cls.vocab_size * cls.embed_dim
        head_dim = cls.embed_dim // cls.num_heads

        q_proj = cls.embed_dim * (cls.num_heads * head_dim)
        k_proj = cls.embed_dim * (cls.num_kv_heads * head_dim)
        v_proj = cls.embed_dim * (cls.num_kv_heads * head_dim)
        out_proj = (cls.num_heads * head_dim) * cls.embed_dim
        attn = q_proj + k_proj + v_proj + out_proj

        mlp = 2 * (cls.embed_dim * cls.intermediate_dim)
        norms = 2 * cls.embed_dim
        per_block = attn + mlp + norms
        all_blocks = cls.num_layers * per_block

        final_norm = cls.embed_dim
        lm_head = cls.vocab_size * cls.embed_dim

        total_tied = tok_emb + all_blocks + final_norm
        total_untied = total_tied + lm_head

        # LoRA parameters across target projections
        # q_proj: 2 * lora_rank * embed_dim; v_proj: 2 * lora_rank * (num_kv_heads * head_dim)
        lora_params = cls.num_layers * (
            (2 * cls.lora_rank * cls.embed_dim) +
            (cls.lora_rank * cls.embed_dim + cls.lora_rank * (cls.num_kv_heads * head_dim))
        )

        kv_cache_bytes = 2 * cls.num_layers * cls.num_kv_heads * head_dim * cls.block_size * 2
        kv_cache_mb = kv_cache_bytes / (1024 * 1024)

        return {
            'token_emb': tok_emb,
            'per_block': per_block,
            'all_blocks': all_blocks,
            'total_tied': total_tied,
            'total_untied': total_untied,
            'lora_params': lora_params,
            'lora_percent': (lora_params / total_untied) * 100,
            'kv_cache_mb': kv_cache_mb,
        }


In [ ]:
%%writefile /content/ss_100million/src/tokenizer.py
"""
Multilingual 10,000-Vocabulary Byte-Pair Encoding (BPE) Tokenizer.
Covers Bengali + English + Mathematics and Logic.
Uses HuggingFace tokenizers ByteLevel BPE for lossless tokenization.
"""

import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders


class MultilingualTokenizer:
    """10,000-Vocab Byte-Pair Encoding (BPE) Tokenizer for Bangla + English + Mathematics."""

    PAD = '<PAD>'
    UNK = '<UNK>'
    BOS = '<BOS>'
    EOS = '<EOS>'
    SYSTEM = '<|system|>'
    USER = '<|user|>'
    ASSISTANT = '<|assistant|>'
    MATH = '<|math|>'

    SPECIAL_TOKENS = [PAD, UNK, BOS, EOS, SYSTEM, USER, ASSISTANT, MATH]

    def __init__(self, vocab_size=10000):
        self.target_vocab_size = vocab_size
        self._tokenizer = None
        self.pad_id = 0
        self.unk_id = 1
        self.bos_id = 2
        self.eos_id = 3
        self.system_id = 4
        self.user_id = 5
        self.assistant_id = 6
        self.math_id = 7

    @property
    def vocab_size(self):
        if self._tokenizer is not None:
            return max(self._tokenizer.get_vocab_size(), self.target_vocab_size)
        return self.target_vocab_size

    def build_vocab(self, text_or_filepath):
        """Train 10,000-vocabulary BPE on Multilingual & Math corpus."""
        print(f"⚡ Training real {self.target_vocab_size:,}-vocab BPE tokenizer (Bangla + English + Math)...")
        
        temp_file = None
        if os.path.exists(text_or_filepath):
            train_file = text_or_filepath
        else:
            temp_file = "temp_corpus_for_tokenizer.txt"
            with open(temp_file, "w", encoding="utf-8") as f:
                f.write(text_or_filepath)
            train_file = temp_file

        tok = Tokenizer(models.BPE(unk_token=self.UNK))
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tok.decoder = decoders.ByteLevel()

        trainer = trainers.BpeTrainer(
            vocab_size=self.target_vocab_size,
            special_tokens=self.SPECIAL_TOKENS,
            min_frequency=1,
            show_progress=False
        )

        tok.train([train_file], trainer)
        self._tokenizer = tok

        if temp_file and os.path.exists(temp_file):
            os.remove(temp_file)

        print(f"✓ Real {self.target_vocab_size:,}-Vocab BPE Tokenizer successfully built!")
        print(f"  - Actual Vocab Size: {self._tokenizer.get_vocab_size():,} tokens")
        return self

    def encode(self, text, add_bos=False, add_eos=False):
        """Encode text string into token IDs."""
        if self._tokenizer is None:
            raise RuntimeError("Tokenizer is not trained or loaded. Call build_vocab() or load().")

        encoded = self._tokenizer.encode(text)
        ids = list(encoded.ids)

        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]

        return ids

    def decode(self, ids, skip_special_tokens=True):
        """Decode token IDs back into text."""
        if self._tokenizer is None:
            raise RuntimeError("Tokenizer is not trained or loaded. Call build_vocab() or load().")

        clean_ids = []
        for i in ids:
            val = int(i)
            if skip_special_tokens and val in [self.pad_id, self.unk_id, self.bos_id, self.eos_id]:
                continue
            clean_ids.append(val)

        return self._tokenizer.decode(clean_ids)

    def save(self, filepath="tokenizer.json"):
        """Save tokenizer configuration to json file."""
        if self._tokenizer is None:
            raise RuntimeError("Tokenizer is not trained or loaded.")
        self._tokenizer.save(filepath)
        print(f"✓ Tokenizer saved to {filepath}")

    def load(self, filepath="tokenizer.json"):
        """Load tokenizer from json file."""
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Tokenizer file not found: {filepath}")
        self._tokenizer = Tokenizer.from_file(filepath)
        print(f"✓ Tokenizer loaded from {filepath} (Vocab: {self.vocab_size:,})")
        return self


# Backward compatibility alias
BengaliTokenizer = MultilingualTokenizer


In [ ]:
%%writefile /content/ss_100million/src/model.py
"""
Modern Decoder-Only Transformer with:
- RoPE (Rotary Position Embeddings)
- GQA (Grouped-Query Attention for 75% KV-cache reduction)
- RMSNorm
- PyTorch SDPA (FlashAttention)
- 2048 Context Length (~50 Million parameters)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.config import GPTConfig


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * norm


def precompute_freqs_cis(dim, end, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cos(freqs)
    freqs_sin = torch.sin(freqs)
    return freqs_cos, freqs_sin


def apply_rotary_emb(x, cos, sin):
    B, H, T, D = x.shape
    x1 = x[..., : D // 2]
    x2 = x[..., D // 2 :]
    cos = cos[:T, :].unsqueeze(0).unsqueeze(1)
    sin = sin[:T, :].unsqueeze(0).unsqueeze(1)
    rx1 = x1 * cos - x2 * sin
    rx2 = x1 * sin + x2 * cos
    return torch.cat([rx1, rx2], dim=-1)


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x.unsqueeze(2).expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.num_heads
        self.n_kv_heads = config.num_kv_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.head_dim = config.embed_dim // config.num_heads
        self.dropout = config.dropout

        self.q_proj = nn.Linear(config.embed_dim, self.n_heads * self.head_dim, bias=config.bias)
        self.k_proj = nn.Linear(config.embed_dim, self.n_kv_heads * self.head_dim, bias=config.bias)
        self.v_proj = nn.Linear(config.embed_dim, self.n_kv_heads * self.head_dim, bias=config.bias)
        self.out_proj = nn.Linear(self.n_heads * self.head_dim, config.embed_dim, bias=config.bias)
        self.resid_drop = nn.Dropout(config.dropout)

    def forward(self, x, cos, sin):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rotary_emb(q, cos, sin)
        k = apply_rotary_emb(k, cos, sin)

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.embed_dim, config.intermediate_dim, bias=config.bias)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(config.intermediate_dim, config.embed_dim, bias=config.bias)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.drop(self.fc2(self.act(self.fc1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.embed_dim)
        self.attn = GroupedQueryAttention(config)
        self.norm2 = RMSNorm(config.embed_dim)
        self.mlp = MLP(config)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.mlp(self.norm2(x))
        return x


class BengaliGPT(nn.Module):
    """50 Million Parameter Bengali Model with 2048 Context Length."""
    def __init__(self, config=None):
        super().__init__()
        if config is None: config = GPTConfig()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.embed_dim)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_layers)])
        self.norm_f = RMSNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        self.tok_emb.weight = self.lm_head.weight

        head_dim = config.embed_dim // config.num_heads
        cos, sin = precompute_freqs_cis(head_dim, config.block_size, config.rope_theta)
        self.register_buffer("freqs_cos", cos, persistent=False)
        self.register_buffer("freqs_sin", sin, persistent=False)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: torch.nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            torch.nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, f"Sequence length {T} exceeds {self.config.block_size}"

        cos = self.freqs_cos[:T].to(idx.device)
        sin = self.freqs_sin[:T].to(idx.device)

        x = self.drop(self.tok_emb(idx))

        for block in self.blocks:
            x = block(x, cos, sin)

        x = self.norm_f(x)
        logits = self.lm_head(x if targets is not None else x[:, [-1], :])

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=250, temperature=0.75, top_k=40, repetition_penalty=1.2, eos_id=3):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            # Repetition penalty প্রয়োগ
            if repetition_penalty != 1.0:
                for b in range(logits.shape[0]):
                    for token_id in set(idx[b].tolist()):
                        logits[b, token_id] /= repetition_penalty

            logits = logits / max(temperature, 1e-5)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            next_token = torch.multinomial(F.softmax(logits, -1), 1)
            idx = torch.cat([idx, next_token], dim=1)

            # EOS টোকেন আসলে জেনারেশন থামানো
            if eos_id is not None and (next_token == eos_id).all():
                break

        return idx


In [ ]:
%%writefile /content/ss_100million/src/lora.py
"""
Custom LoRA (Low-Rank Adaptation) and QLoRA implemented from scratch.
Zero dependency on Hugging Face PEFT library. Pure PyTorch implementation.
Optimized for 50M Parameter Bengali GPT with 2048 context length.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class LoRALinear(nn.Module):
    """
    Pure PyTorch LoRA / QLoRA Linear Layer.
    Wraps an existing linear layer or creates a new one:
    - Freezes base model weights in FP16 / FP32 (< 100 MB VRAM for 50M model)
    - Adds trainable low-rank matrices A and B (~0.21M trainable params)
    - 100% immune to bitsandbytes CUDA version mismatches
    """

    def __init__(self, in_features_or_module, out_features=None, rank=8, alpha=16, dropout=0.05, bias=False):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        if isinstance(in_features_or_module, nn.Linear):
            # Wrap existing linear layer directly without fragile weight copying
            base_linear = in_features_or_module
            self.in_features = base_linear.in_features
            self.out_features = base_linear.out_features
            self.linear = base_linear
        else:
            self.in_features = in_features_or_module
            self.out_features = out_features
            self.linear = nn.Linear(self.in_features, self.out_features, bias=bias)

        # Freeze base weights
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # Trainable LoRA low-rank matrices
        weight_param = self.linear.weight
        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=weight_param.device, dtype=weight_param.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=weight_param.device, dtype=weight_param.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x):
        base_out = self.linear(x)
        # Ensure dtype match during mixed precision training (AMP FP16)
        x_drop = self.dropout(x)
        lora_out = (x_drop @ self.lora_A.t().to(x_drop.dtype)) @ self.lora_B.t().to(x_drop.dtype)
        return base_out + self.scale * lora_out

    def merge(self):
        """Merges LoRA weights back into the base linear layer for zero-latency inference."""
        with torch.no_grad():
            self.linear.weight.data += self.scale * (self.lora_B @ self.lora_A).to(self.linear.weight.dtype)


class QLoRALinear(LoRALinear):
    """
    QLoRA Layer: High-performance LoRA with frozen FP16 base weights.
    For a 50M parameter model, base weights consume only ~98 MB VRAM!
    Provides complete compatibility, stability, and maximum speed on NVIDIA T4.
    """
    def __init__(self, in_features_or_module, out_features=None, rank=8, alpha=16, dropout=0.05, bias=False, use_bnb=False):
        super().__init__(in_features_or_module, out_features, rank=rank, alpha=alpha, dropout=dropout, bias=bias)
        self.linear.weight.requires_grad = False


def apply_lora(model, rank=8, alpha=16, dropout=0.05, target_modules=['q_proj', 'v_proj'], use_qlora=False):
    """
    Recursively traverse model and replace target linear layers with LoRA / QLoRA layers.
    Freezes all base model weights.
    Returns:
        model: Modified model with LoRA applied
        trainable_params: Number of trainable LoRA parameters
    """
    # 1. Freeze all parameters in the base model
    for param in model.parameters():
        param.requires_grad = False

    replaced_count = 0

    # 2. Helper to recursively replace modules
    def _replace_modules(module, prefix=""):
        nonlocal replaced_count
        for child_name, child_module in list(module.named_children()):
            full_name = f"{prefix}.{child_name}" if prefix else child_name

            if any(target == child_name for target in target_modules) and isinstance(child_module, nn.Linear):
                # Seamless wrap: preserve device, dtype, weights and bias
                lora_layer = LoRALinear(child_module, rank=rank, alpha=alpha, dropout=dropout)
                setattr(module, child_name, lora_layer)
                replaced_count += 1
            else:
                _replace_modules(child_module, full_name)

    _replace_modules(model)

    # 3. Parameter count summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("=" * 60)
    mode_name = "QLoRA (Frozen Base FP16 + Trainable Adapters)" if use_qlora else "LoRA (PyTorch Native)"
    print(f"Applied {mode_name} to model:")
    print(f"  - Replaced linear layers: {replaced_count}")
    print(f"  - Target modules: {target_modules}")
    print(f"  - LoRA Rank: {rank}, Alpha: {alpha}, Dropout: {dropout}")
    print(f"  - Total parameters: {total_params:,}")
    pct = 100.0 * trainable_params / total_params if total_params > 0 else 0.0
    print(f"  - Trainable parameters: {trainable_params:,} ({pct:.2f}%)")
    pass_flag = "PASS" if pct < 1.0 else "INFO"
    print(f"  - Trainable parameters < 1%: [{pass_flag}] ({pct:.2f}% of full model)")
    print("=" * 60)

    return model, trainable_params


def get_lora_state_dict(model):
    """Extract state dictionary containing only trainable LoRA parameters."""
    return {k: v for k, v in model.state_dict().items() if "lora_" in k}


def save_lora(model, filepath):
    """Save only the LoRA adapter parameters to file."""
    lora_dict = get_lora_state_dict(model)
    torch.save(lora_dict, filepath)
    print(f"Saved LoRA adapter weights ({len(lora_dict)} tensors) to {filepath}")


def load_lora(model, filepath):
    """Load LoRA adapter parameters into model."""
    state_dict = torch.load(filepath, map_location="cpu")
    model_state = model.state_dict()
    for k, v in state_dict.items():
        if k in model_state:
            model_state[k].copy_(v)
    print(f"Loaded LoRA adapter weights from {filepath}")
    return model


In [ ]:
%%writefile /content/ss_100million/src/dataset.py
"""
Memory-efficient dataset loader with uint16 binary memmap caching.
Supports 700MB+ / 10GB+ corpora on Colab Free Tier without RAM crashes (< 100MB RAM!).
Supports configurable vocab_size for multilingual (10k) and Bengali-only (5k) tokenizers.
"""

import os
import numpy as np
import torch
from src.tokenizer import MultilingualTokenizer


class BengaliDataset:
    """
    Streaming, memory-mapped dataset for massive corpora.
    Converts corpus.txt into a compact uint16 binary file in chunks without RAM spikes.
    Uses np.memmap for zero-RAM batch sampling.
    Supports configurable vocab_size — pass vocab_size=10000 for multilingual 100M model.
    """

    def __init__(self, corpus_path="data/corpus.txt", bin_cache_dir=None,
                 tokenizer=None, split_ratio=0.9, block_size=2048, vocab_size=10000):
        self.corpus_path = corpus_path
        self.block_size = block_size
        self.split_ratio = split_ratio

        if not os.path.exists(corpus_path):
            raise FileNotFoundError(f"Corpus file not found: {corpus_path}")

        file_size_mb = os.path.getsize(corpus_path) / (1024 * 1024)
        print(f"📖 কর্পাস ফাইল: {corpus_path} ({file_size_mb:.1f} MB)")

        # ─── টোকেনাইজার লোড বা তৈরি ───────────────────────────────────────────
        if tokenizer is None:
            self.tokenizer = MultilingualTokenizer(vocab_size=vocab_size)

            # অগ্রাধিকার ক্রমে টোকেনাইজার খোঁজা:
            # ১. লোকাল tokenizer.json (Colab /content/ss_100million/tokenizer.json)
            # ২. ড্রাইভের 100M চেকপয়েন্ট ফোল্ডার
            # ৩. ড্রাইভের 50M চেকপয়েন্ট ফোল্ডার (Fallback)
            # ৪. স্ক্র্যাচ থেকে নতুন তৈরি করা
            search_paths = [
                "tokenizer.json",
                "/content/drive/MyDrive/bengali_gpt_100m_checkpoints/tokenizer.json",
                "/content/drive/MyDrive/bengali_gpt_50m_checkpoints/tokenizer.json",
            ]
            loaded = False
            for tok_path in search_paths:
                if os.path.exists(tok_path):
                    self.tokenizer.load(tok_path)
                    loaded = True
                    break

            if not loaded:
                # কোনো টোকেনাইজার পাওয়া যায়নি → নতুন করে তৈরি করো
                print(f"⚡ কোনো পূর্ববর্তী tokenizer.json পাওয়া যায়নি। কর্পাস থেকে {vocab_size:,}-vocab BPE তৈরি হচ্ছে...")
                self.tokenizer.build_vocab(corpus_path)
                self.tokenizer.save("tokenizer.json")
                print(f"✓ নতুন {vocab_size:,}-vocab টোকেনাইজার তৈরি ও সেভ সম্পন্ন!")
        else:
            self.tokenizer = tokenizer

        # ─── লোকাল Colab ডিস্কে বাইনারি ক্যাশ ফাইল ────────────────────────────
        # গুগল ড্রাইভের পাথ হলে Colab লোকাল ডিস্কে ক্যাশ করো (ড্রাইভ I/O অনেক ধীর)
        if bin_cache_dir is None:
            corpus_dir = os.path.dirname(corpus_path)
            if corpus_dir.startswith("/content/drive"):
                bin_cache_dir = "/content/ss_100million/data"
            elif not corpus_dir:
                bin_cache_dir = "data"
            else:
                bin_cache_dir = corpus_dir

        os.makedirs(bin_cache_dir, exist_ok=True)
        bin_file = os.path.join(bin_cache_dir, "corpus_tokens.bin")

        # ─── চাঙ্কড স্ট্রিমিং টোকেনাইজেশন (RAM মাত্র ~৫০ MB ব্যবহার হয়!) ──────
        if not os.path.exists(bin_file) or os.path.getsize(bin_file) == 0:
            print(f"⚡ মেমরি-সেফ চাঙ্কড টোকেনাইজেশন শুরু হচ্ছে (RAM সুরক্ষিত থাকবে)...")
            print(f"  → ক্যাশ লোকেশন: {bin_file}")
            total_tokens = 0
            chunk_lines = []
            chunk_chars = 0
            CHUNK_LIMIT = 500_000  # প্রতি চাঙ্কে ~৫ লক্ষ অক্ষর

            with open(corpus_path, "r", encoding="utf-8") as f_in, open(bin_file, "wb") as f_out:
                for line in f_in:
                    chunk_lines.append(line)
                    chunk_chars += len(line)
                    if chunk_chars >= CHUNK_LIMIT:
                        text_chunk = "".join(chunk_lines)
                        ids = self.tokenizer.encode(text_chunk)
                        np.array(ids, dtype=np.uint16).tofile(f_out)
                        total_tokens += len(ids)
                        chunk_lines = []
                        chunk_chars = 0
                        if total_tokens % 2_000_000 < len(ids):
                            print(f"  ✍️ {total_tokens:,} টোকেন প্রক্রিয়াজাত (RAM নিরাপদ)...")

                if chunk_lines:
                    text_chunk = "".join(chunk_lines)
                    ids = self.tokenizer.encode(text_chunk)
                    np.array(ids, dtype=np.uint16).tofile(f_out)
                    total_tokens += len(ids)

            print(f"✓ বাইনারি ক্যাশ সম্পন্ন: {bin_file}")
            print(f"  → মোট টোকেন: {total_tokens:,}")
        else:
            print(f"✓ বিদ্যমান বাইনারি ক্যাশ লোড হচ্ছে: {bin_file}")

        # ─── Memmap দিয়ে ডিস্ক থেকে সরাসরি মেমরি ম্যাপিং ─────────────────────
        self.data = np.memmap(bin_file, dtype=np.uint16, mode="r")
        self.total_tokens = len(self.data)

        self.train_len = int(self.split_ratio * self.total_tokens)
        self.val_len = self.total_tokens - self.train_len

        print(f"⚡ Dataset Memmap Ready: {self.total_tokens:,} Tokens")
        print(f"  - Train: {self.train_len:,} টোকেন ({self.split_ratio*100:.0f}%)")
        print(f"  - Val:   {self.val_len:,} টোকেন ({(1-self.split_ratio)*100:.0f}%)")
        print(f"  - Sequences (at {block_size} context): ~{self.train_len // block_size:,}")

    def get_batch(self, split="train", batch_size=8, device="cpu"):
        if split == "train":
            max_start = self.train_len - self.block_size - 1
            offset = 0
        else:
            max_start = self.val_len - self.block_size - 1
            offset = self.train_len

        if max_start <= 0:
            raise ValueError("Dataset too small for context length.")

        ix = np.random.randint(offset, offset + max_start, size=(batch_size,))
        x_list = [torch.from_numpy((self.data[i : i + self.block_size]).astype(np.int64)) for i in ix]
        y_list = [torch.from_numpy((self.data[i + 1 : i + 1 + self.block_size]).astype(np.int64)) for i in ix]

        x = torch.stack(x_list)
        y = torch.stack(y_list)

        if device != "cpu":
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

        return x, y


In [ ]:
%%writefile /content/ss_100million/src/train.py
"""
Training pipeline for 50M parameter Bengali GPT with 2048 context length.
Features: Gradient accumulation, Mixed Precision, QLoRA, and Auto-Resume.
"""

import os
import math
import time
import glob
import torch
from src.config import GPTConfig
from src.model import BengaliGPT
from src.dataset import BengaliDataset
from src.lora import apply_lora, save_lora


def _lr(step, config):
    if step < config.warmup_iters:
        return config.learning_rate * step / config.warmup_iters
    decay = (step - config.warmup_iters) / max(1, config.max_iters - config.warmup_iters)
    coeff = 0.5 * (1 + math.cos(math.pi * decay))
    return config.min_lr + coeff * (config.learning_rate - config.min_lr)


@torch.no_grad()
def _eval_loss(model, ds, config):
    model.eval()
    losses = {}
    for split in ('train', 'val'):
        L = []
        for _ in range(config.eval_iters):
            x, y = ds.get_batch(split, config.batch_size, config.device)
            with torch.amp.autocast('cuda', dtype=torch.float16) if 'cuda' in config.device else torch.nullcontext():
                _, loss = model(x, y)
            L.append(loss.item())
        losses[split] = sum(L) / len(L)
    model.train()
    return losses


def _find_last_checkpoint(ckpt_dir, prefix):
    files = glob.glob(os.path.join(ckpt_dir, f"{prefix}_step_*.pt"))
    if not files:
        return None, 0

    def extract_step(path):
        try:
            return int(path.split('_step_')[-1].replace('.pt', ''))
        except ValueError:
            return -1

    # সংখ্যাগতভাবে (Numerically) সর্বোচ্চ স্টেপটি বাছাই করুন
    files = sorted(files, key=extract_step)
    latest = files[-1]
    step = extract_step(latest)
    if step <= 0:
        return None, 0
    return latest, step


def _save_checkpoint(model, optimizer, step, val_loss, config, prefix, is_best=False):
    ckpt = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'step': step,
        'val_loss': val_loss,
        'config': {k: v for k, v in vars(config).items() if not k.startswith('_')},
    }
    path = os.path.join(config.checkpoint_dir, f"{prefix}_step_{step}.pt")
    torch.save(ckpt, path)
    print(f"  💾 চেকপয়েন্ট ড্রাইভে সেভ হয়েছে: {path}")
    if is_best:
        best_path = os.path.join(config.checkpoint_dir, f"{prefix}_best.pt")
        torch.save(ckpt, best_path)
        print(f"  🌟 সেরা মডেল ড্রাইভে আপডেট হয়েছে: {best_path}")


def train_qlora(config=None, corpus_path="data/corpus.txt", base_checkpoint=None):
    if config is None:
        config = GPTConfig()
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    torch.manual_seed(config.seed)

    # মডেলের প্যারামিটার সংখ্যা অনুযায়ী প্রিফিক্স নির্ধারণ করা
    total_params = sum(p.numel() for p in __import__('src.model', fromlist=['BengaliGPT']).BengaliGPT(config).parameters())
    param_m = round(total_params / 1e6)
    PREFIX = f"qlora_{param_m}m"

    print("=" * 65)
    print(f"🚀 {param_m}M MULTILINGUAL GPT — QLoRA ফাইন-টিউনিং শুরু হচ্ছে ({config.block_size} কনটেক্সট লেন্থ)")
    print(f"ডিভাইস: {config.device} | ব্যাচ: {config.batch_size} (Grad Accum: {config.gradient_accumulation_steps}) | সর্বমোট স্টেপ: {config.max_iters}")
    print("=" * 65)

    # vocab_size কনফিগ থেকে BengaliDataset-এ পাঠানো হচ্ছে (5k বা 10k উভয়ক্ষেত্রে সঠিক থাকবে)
    ds = BengaliDataset(corpus_path, block_size=config.block_size, vocab_size=config.vocab_size)
    config.vocab_size = ds.tokenizer.vocab_size
    ds.tokenizer.save(os.path.join(config.checkpoint_dir, "tokenizer.json"))

    model = BengaliGPT(config).to(config.device)

    # যদি কোনো প্রি-ট্রেইন্ড বেস মডেল দেওয়া থাকে, তার ওজন লোড করুন
    if base_checkpoint and os.path.exists(base_checkpoint):
        print(f"📥 বেস মডেল থেকে জ্ঞান লোড হচ্ছে: {base_checkpoint}")
        ckpt_base = torch.load(base_checkpoint, map_location=config.device)
        model.load_state_dict(ckpt_base['model'], strict=False)
        print("✓ বেস মডেলের ওজন সফলভাবে লোড হয়েছে!")

    model, trainable = apply_lora(
        model,
        rank=config.lora_rank,
        alpha=config.lora_alpha,
        dropout=config.lora_dropout,
        target_modules=config.lora_target_modules,
        use_qlora=True
    )

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
        weight_decay=config.weight_decay
    )
    device_type = 'cuda' if 'cuda' in config.device else 'cpu'
    scaler = torch.amp.GradScaler('cuda', enabled=('cuda' in config.device))

    # Auto-resume from Drive
    last_ckpt, start_step = _find_last_checkpoint(config.checkpoint_dir, PREFIX)
    if last_ckpt:
        ckpt = torch.load(last_ckpt, map_location=config.device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        print(f"🔄 পূর্ববর্তী চেকপয়েন্ট পাওয়া গেছে! {start_step} নম্বর স্টেপ থেকে ট্রেনিং চালু হচ্ছে...")
    else:
        print("🌱 কোনো পুরানো চেকপয়েন্ট নেই, শুরু থেকে (স্টেপ ১) ট্রেনিং শুরু হচ্ছে...")

    best_val = float('inf')
    t0 = time.time()

    for step in range(start_step + 1, config.max_iters + 1):
        lr = _lr(step, config)
        for g in optimizer.param_groups:
            g['lr'] = lr

        optimizer.zero_grad(set_to_none=True)

        accum_loss = 0.0
        # Gradient accumulation loop for 2048 context
        for _ in range(config.gradient_accumulation_steps):
            x, y = ds.get_batch('train', config.batch_size, config.device)
            with torch.amp.autocast('cuda', dtype=torch.float16) if 'cuda' in config.device else torch.nullcontext():
                _, loss = model(x, y)
                loss = loss / config.gradient_accumulation_steps
            accum_loss += loss.item()
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], config.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        # প্রতি ২৫ স্টেপে ভিজ্যুয়াল প্রগ্রেস বার সহ লাইভ আপডেট
        if step % 25 == 0 and step % config.eval_interval != 0:
            elapsed = time.time() - t0
            steps_done = step - start_step
            speed = elapsed / max(1, steps_done)
            eta_mins = (config.max_iters - step) * speed / 60
            pct = (step / config.max_iters) * 100
            bar_len = 20
            filled = int(bar_len * step / config.max_iters)
            bar = '█' * filled + '░' * (bar_len - filled)
            print(f"  [{bar}] {pct:5.1f}% | Step {step:4d}/{config.max_iters} | Loss: {accum_loss:.4f} | LR: {lr:.2e} | গতি: {speed:.2f}s/step | বাকি: ~{eta_mins:.1f} মি.")

        if step % config.eval_interval == 0 or step == config.max_iters:
            losses = _eval_loss(model, ds, config)
            elapsed = time.time() - t0
            pct = (step / config.max_iters) * 100
            bar_len = 20
            filled = int(bar_len * step / config.max_iters)
            bar = '█' * filled + '░' * (bar_len - filled)
            print("─" * 70)
            print(f"  [{bar}] {pct:5.1f}% | Step {step:5d}/{config.max_iters} | Train: {losses['train']:.4f} | Val: {losses['val']:.4f} | সময়: {elapsed/60:.1f} মি.")
            print("─" * 70)
            is_best = losses['val'] < best_val
            if is_best:
                best_val = losses['val']
            _save_checkpoint(model, optimizer, step, losses['val'], config, PREFIX, is_best)
        elif step % config.save_interval == 0:
            _save_checkpoint(model, optimizer, step, None, config, PREFIX)

    print(f"🎉 {param_m}M মডেলের ট্রেনিং সফলভাবে সম্পন্ন! সেরা ভ্যালিডেশন লস: {best_val:.4f}")
    return model


def train_pretrain(config=None, corpus_path="data/corpus.txt"):
    if config is None: config = GPTConfig()
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    torch.manual_seed(config.seed)

    # মডেলের প্যারামিটার সংখ্যা অনুযায়ী প্রিফিক্স নির্ধারণ
    total_params = sum(p.numel() for p in BengaliGPT(config).parameters())
    param_m = round(total_params / 1e6)
    PREFIX = f"pretrain_{param_m}m"

    print("=" * 65)
    print(f"🚀 {param_m}M MULTILINGUAL GPT — FULL PRETRAINING ({config.block_size} কনটেক্সট লেন্থ)")
    print("=" * 65)

    ds = BengaliDataset(corpus_path, block_size=config.block_size, vocab_size=config.vocab_size)
    config.vocab_size = ds.tokenizer.vocab_size
    ds.tokenizer.save(os.path.join(config.checkpoint_dir, "tokenizer.json"))

    model = BengaliGPT(config).to(config.device)
    total = sum(p.numel() for p in model.parameters())
    print(f"✓ সর্বমোট প্যারামিটার: {total:,} (~{total/1e6:.2f}M)")

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, betas=(config.beta1, config.beta2), weight_decay=config.weight_decay)
    device_type = 'cuda' if 'cuda' in config.device else 'cpu'
    scaler = torch.amp.GradScaler('cuda', enabled=('cuda' in config.device))

    last_ckpt, start_step = _find_last_checkpoint(config.checkpoint_dir, PREFIX)
    if last_ckpt:
        ckpt = torch.load(last_ckpt, map_location=config.device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        print(f"🔄 পূর্ববর্তী চেকপয়েন্ট পাওয়া গেছে: {start_step} নম্বর স্টেপ থেকে রিজ্যুম হচ্ছে...")
    else:
        print("🌱 শুরু থেকে স্ক্র্যাচ ট্রেনিং হচ্ছে...")

    best_val = float('inf')
    t0 = time.time()

    for step in range(start_step + 1, config.max_iters + 1):
        lr = _lr(step, config)
        for g in optimizer.param_groups: g['lr'] = lr

        accum_loss = 0.0
        for _ in range(config.gradient_accumulation_steps):
            x, y = ds.get_batch('train', config.batch_size, config.device)
            with torch.amp.autocast('cuda', dtype=torch.float16) if 'cuda' in config.device else torch.nullcontext():
                _, loss = model(x, y)
                loss = loss / config.gradient_accumulation_steps
            accum_loss += loss.item()
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        # প্রতি ২৫ স্টেপে ভিজ্যুয়াল প্রগ্রেস বার সহ লাইভ আপডেট
        if step % 25 == 0 and step % config.eval_interval != 0:
            elapsed = time.time() - t0
            steps_done = step - start_step
            speed = elapsed / max(1, steps_done)
            eta_mins = (config.max_iters - step) * speed / 60
            pct = (step / config.max_iters) * 100
            bar_len = 20
            filled = int(bar_len * step / config.max_iters)
            bar = '█' * filled + '░' * (bar_len - filled)
            print(f"  [{bar}] {pct:5.1f}% | Step {step:4d}/{config.max_iters} | Loss: {accum_loss:.4f} | LR: {lr:.2e} | গতি: {speed:.2f}s/step | বাকি: ~{eta_mins:.1f} মি.")

        if step % config.eval_interval == 0 or step == config.max_iters:
            losses = _eval_loss(model, ds, config)
            elapsed = time.time() - t0
            pct = (step / config.max_iters) * 100
            bar_len = 20
            filled = int(bar_len * step / config.max_iters)
            bar = '█' * filled + '░' * (bar_len - filled)
            print("─" * 70)
            print(f"  [{bar}] {pct:5.1f}% | Step {step:5d}/{config.max_iters} | Train: {losses['train']:.4f} | Val: {losses['val']:.4f} | সময়: {elapsed/60:.1f} মি.")
            print("─" * 70)
            is_best = losses['val'] < best_val
            if is_best: best_val = losses['val']
            _save_checkpoint(model, optimizer, step, losses['val'], config, PREFIX, is_best)
        elif step % config.save_interval == 0:
            _save_checkpoint(model, optimizer, step, None, config, PREFIX)

    print(f"🎉 {param_m}M মডেলের ফুল প্রি-ট্রেইনিং সম্পন্ন! সেরা লস: {best_val:.4f}")
    return model


## 🚀 ধাপ ৬: ১০০M বেস মডেলের ট্রেনিং (৪০৯৬ কনটেক্সট লেন্থ, ~৪.৫ GB VRAM)

- মাইক্রো ব্যাচ: ৪ (সিকোয়েন্স দৈর্ঘ্য ৪০৯৬ টোকেন)
- গ্রেডিয়েন্ট এক্যুমুলেশন: ৪ (ভার্চুয়াল ব্যাচ = ১৬ সিকোয়েন্স = ৬৫,৫৩৬ টোকেন/স্টেপ!)
- স্বয়ংক্রিয় সেভ ও রিজ্যুম: ড্রাইভের `/content/drive/MyDrive/bengali_gpt_100m_checkpoints` ফোল্ডারে


In [ ]:
import sys, os
sys.path.insert(0, '/content/ss_100million')
os.chdir('/content/ss_100million')

for mod in list(sys.modules.keys()):
    if mod.startswith('src'): del sys.modules[mod]

from src.config import GPTConfig
from src.train import train_qlora

config = GPTConfig()
config.block_size                  = 4096   # ৪০৯৬ কনটেক্সট লেন্থ (4k Context!)
config.batch_size                  = 4      # মাইক্রো ব্যাচ সাইজ (4x4096)
config.gradient_accumulation_steps = 4      # ভার্চুয়াল ব্যাচ সাইজ = ১৬ (৬৫,৫৩৬ টোকেন/স্টেপ)
config.max_iters                   = 5000   # ৫,০০০ স্টেপ
config.save_interval               = 500    # প্রতি ৫০০ স্টেপে চেকপয়েন্ট ড্রাইভে সেভ হবে
config.eval_interval               = 500    # প্রতি ৫০০ স্টেপে ভ্যালিডেশন লস পরীক্ষা
config.checkpoint_dir              = '/content/drive/MyDrive/bengali_gpt_100m_checkpoints'

corpus_path = '/content/drive/MyDrive/corpus.txt'
print('=' * 65)
print('🚀 ১০০M MULTILINGUAL BASE MODEL ট্রেনিং শুরু হচ্ছে (4k Context)...')
print('=' * 65)

model = train_qlora(config=config, corpus_path=corpus_path)


## 💬 ধাপ ৭: মাল্টিলিঙ্গুয়াল ও ম্যাথ টেস্ট (বাংলা + English + Math — Dataset ছাড়া)


In [ ]:
import torch, os, sys, glob
sys.path.insert(0, '/content/ss_100million')
os.chdir('/content/ss_100million')

for mod in list(sys.modules.keys()):
    if mod.startswith('src'): del sys.modules[mod]

from src.config import GPTConfig
from src.model import BengaliGPT
from src.tokenizer import MultilingualTokenizer
from src.lora import apply_lora

CKPT_DIR = '/content/drive/MyDrive/bengali_gpt_100m_checkpoints'
config = GPTConfig()

tok = MultilingualTokenizer(vocab_size=10000)
tok_path = os.path.join(CKPT_DIR, 'tokenizer.json') if os.path.exists(os.path.join(CKPT_DIR, 'tokenizer.json')) else 'tokenizer.json'
tok.load(tok_path)
config.vocab_size = tok.vocab_size

model = BengaliGPT(config)
model, _ = apply_lora(model, rank=config.lora_rank, alpha=config.lora_alpha,
                      target_modules=config.lora_target_modules, use_qlora=True)

def extract_step(path):
    try: return int(path.split('_step_')[-1].replace('.pt', ''))
    except: return -1

best = os.path.join(CKPT_DIR, 'qlora_96m_best.pt')
if not os.path.exists(best):
    best_candidates = glob.glob(os.path.join(CKPT_DIR, '*_best.pt'))
    best = best_candidates[0] if best_candidates else None

all_step_ckpts = glob.glob(os.path.join(CKPT_DIR, '*_step_*.pt'))
sorted_ckpts = sorted(all_step_ckpts, key=extract_step)
ckpt_file = best if (best and os.path.exists(best)) else (sorted_ckpts[-1] if sorted_ckpts else None)

if ckpt_file and os.path.exists(ckpt_file):
    ckpt = torch.load(ckpt_file, map_location=config.device)
    model.load_state_dict(ckpt['model'], strict=False)
    print(f'✓ প্রশিক্ষিত ১০০M মডেল সফলভাবে লোড হয়েছে: {ckpt_file}')
else:
    print('⚠️ কোনো চেকপয়েন্ট পাওয়া যায়নি!')

model.to(config.device)
model.eval()

def ask(q, max_tokens=200, temperature=0.75, top_k=40, repetition_penalty=1.2):
    prompt = f'প্রশ্ন: {q}\nউত্তর: '
    ids = tok.encode(prompt, add_bos=True)
    idx = torch.tensor([ids], dtype=torch.long, device=config.device)
    out = model.generate(idx, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k, repetition_penalty=repetition_penalty, eos_id=tok.eos_id)
    return tok.decode(out[0][len(ids):].tolist(), skip_special_tokens=True).strip()

print('\n' + '=' * 60)
print('💬 ১০০M এআই মডেলকে প্রশ্ন করুন (বাংলা, English বা গণিত)')
print('বের হতে চাইলে exit লিখুন')
print('=' * 60)

while True:
    try:
        q = input('\n👤 আপনার প্রশ্ন: ').strip()
        if not q: continue
        if q.lower() in ['exit', 'quit']: break
        print('🤖 উত্তর:', ask(q))
        print('-' * 50)
    except KeyboardInterrupt:
        break
